# Notebook 02 — Knowledge Graph Analysis

Query the materialised KG and visualise accessibility & vulnerability findings.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.utils.rdf_utils import load_graph, sparql_query, count_triples_by_type

# Load all RDF files (run build_kg.py first)
g = load_graph()
print(f'Total triples: {len(g)}')

type_counts = count_triples_by_type(g)
for t, c in list(type_counts.items())[:10]:
    print(f'  {t.split("#")[-1]:30s}  {c}')

In [ ]:
# Vulnerability scores
q = """
PREFIX hkg: <http://healthcare-kg.at/ontology#>
SELECT ?name ?vuln ?risk ?gpPer1000
WHERE {
    ?d a hkg:District ;
       hkg:vulnerabilityScore ?vuln .
    OPTIONAL { ?d hkg:districtName ?name . }
    OPTIONAL { ?d hkg:accessRisk ?risk . }
    OPTIONAL { ?d hkg:gpPer1000 ?gpPer1000 . }
}
ORDER BY DESC(?vuln)
"""
import pandas as pd
rows = sparql_query(g, q)
df = pd.DataFrame(rows)
df['vuln'] = df['vuln'].astype(float)
df['gpPer1000'] = df['gpPer1000'].astype(float)
df['risk'] = df['risk'].str.split('#').str[-1]
df.head(10)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

risk_color = {'HighRisk': '#e74c3c', 'MediumRisk': '#e67e22', 'LowRisk': '#27ae60'}
colors = df['risk'].map(risk_color).fillna('gray')

df_sorted = df.sort_values('vuln', ascending=True)
plt.figure(figsize=(10, 7))
bars = plt.barh(df_sorted['name'], df_sorted['vuln'],
                color=df_sorted['risk'].map(risk_color).fillna('gray'))
plt.axvline(0.5, color='black', linestyle='--', alpha=0.5, label='High risk threshold')
plt.xlabel('Vulnerability Score')
plt.title('District Vulnerability Scores')
plt.legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in risk_color.items()]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('../data/processed/vulnerability_scores.png', dpi=150)
plt.show()

In [ ]:
# Reachability analysis — how many hospitals are within 60 min of each district?
q2 = """
PREFIX hkg: <http://healthcare-kg.at/ontology#>
SELECT ?districtName (COUNT(?hospital) AS ?hospitalCount)
WHERE {
    ?hospital a hkg:Hospital ;
              hkg:reachableIn60min ?district .
    OPTIONAL { ?district hkg:districtName ?districtName . }
}
GROUP BY ?districtName
ORDER BY ?hospitalCount
"""
rows2 = sparql_query(g, q2)
df2 = pd.DataFrame(rows2)
df2['hospitalCount'] = df2['hospitalCount'].astype(int)

plt.figure(figsize=(10, 6))
plt.barh(df2['districtName'], df2['hospitalCount'], color='steelblue')
plt.xlabel('Hospitals reachable within 60 min')
plt.title('Hospital Accessibility by District')
plt.tight_layout()
plt.savefig('../data/processed/hospital_accessibility.png', dpi=150)
plt.show()